In [129]:
%matplotlib qt
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
from datetime import datetime
import glob
from show import *
from utils import *


In [2]:
def remap(data, header, nx=720, ny=1800, **kwargs):
    from transforms import ToSpherical
    from scipy.ndimage import map_coordinates

    view = View.from_header(header).update(**kwargs)
    transform = ~ToSpherical() - view.to_carrington(**kwargs)

    lat_spread = 2
    dlon = 360 / ny
    dlat = lat_spread / (nx - 1)


    grid = np.mgrid[-lat_spread / 2:lat_spread / 2 + dlat / 2:dlat, :360:dlon]
    grid[0] = np.arcsin(grid[0].clip(-1,1)) * 180 / np.pi

    grid, alpha = transform(grid)
    data_ = map_coordinates(np.nan_to_num(data), grid, cval=np.nan)

    return data_, grid, alpha


In [187]:
files = np.array(sorted(glob.glob('/home/ulyanov/data/solo/phi/2026/blos_/*')))
dates = np.array([datetime.fromisoformat(file.split('/')[-1].split('_')[-3]) for file in files])

#t = np.where(np.all([dates > datetime(2026,7,22),
#                     dates < datetime(2026,8,31)], axis=0))[0] # CR2314

#t = np.where(np.all([dates > datetime(2026,8,31),
#                     dates < datetime(2026,9,29)], axis=0))[0] # CR2315

t = np.where(np.all([dates > datetime(2026,7,8),
                     dates < datetime(2026,8,8)], axis=0))[0]

#t = np.where(np.all([dates > datetime(2026,9,14),
#                     dates < datetime(2026,10,13)], axis=0))[0]



files = files[t]
files[0], files[-1], len(files)

(np.str_('/home/ulyanov/data/solo/phi/2026/blos_/phi-fdt-blos_20260708T150009_V202610211128C_0647080501.fits'),
 np.str_('/home/ulyanov/data/solo/phi/2026/blos_/phi-fdt-blos_20260807T120009_V202610201757C_0648070503.fits'),
 126)

In [188]:
N = 0
X, Y, Z, F = 0, 0, 0, 0
X2, Y2, Z2 = 0, 0, 0
XY, XZ, YZ = 0, 0, 0
XF, YF, ZF = 0, 0, 0

for file in files:

    with fits.open(file) as hdul:
        header = hdul[0].header.copy()
        data = hdul[0].data.copy()

    x0, y0 = header['PXBEG2'] - 1, header['PXBEG1'] - 1
    f, grid, alpha = remap(data, header, delta_t=10)

    n = (~np.isnan(f)).astype(float)
    x, y = grid
    x, y = (x + x0) / 1024 - 1, (y + y0) / 1024 - 1

    z = 1 / alpha

    N += n

    X += np.nan_to_num((x - X) * n / N)
    Y += np.nan_to_num((y - Y) * n / N)
    Z += np.nan_to_num((z - Z) * n / N)
    F += np.nan_to_num((f - F) * n / N)

    X2 += np.nan_to_num((x ** 2 - X2) * n / N)
    Y2 += np.nan_to_num((y ** 2 - Y2) * n / N)
    Z2 += np.nan_to_num((z ** 2 - Z2) * n / N)

    XY += np.nan_to_num((x * y - XY) * n / N)
    XZ += np.nan_to_num((x * z - XZ) * n / N)
    YZ += np.nan_to_num((y * z - YZ) * n / N)

    XF += np.nan_to_num((x * f - XF) * n / N)
    YF += np.nan_to_num((y * f - YF) * n / N)
    ZF += np.nan_to_num((z * f - ZF) * n / N)


X[N == 0] = np.nan
Y[N == 0] = np.nan
Z[N == 0] = np.nan
F[N == 0] = np.nan
X2[N == 0] = np.nan
Y2[N == 0] = np.nan
Z2[N == 0] = np.nan
XY[N == 0] = np.nan
XZ[N == 0] = np.nan
YZ[N == 0] = np.nan
XF[N == 0] = np.nan
YF[N == 0] = np.nan
ZF[N == 0] = np.nan
N[N == 0] = np.nan

/tmp/ipykernel_98661/3440083074.py:26: RuntimeWarning: invalid value encountered in divide
  Z += np.nan_to_num((z - Z) * n / N)
/tmp/ipykernel_98661/3440083074.py:31: RuntimeWarning: invalid value encountered in divide
  Z2 += np.nan_to_num((z ** 2 - Z2) * n / N)


In [189]:
X2_ = np.nanmean(X2)
Y2_ = np.nanmean(Y2)
XY_ = np.nanmean(XY)
X_ = np.nanmean(X)
Y_ = np.nanmean(Y)

B = np.moveaxis(np.array([[XZ, YZ, Z]]), (0,1), (-2,-1))
BT = np.swapaxes(B, -1, 2)

A = 1 / np.expand_dims(Z2, (-2,-1))

D = np.array([[X2_, XY_, X_],
              [XY_, Y2_, Y_],
              [X_, Y_, 1]])

H = np.linalg.inv(D - np.nanmean(BT @ B * A, axis=(0,1)))
V = np.nanmean(np.array([XF, YF, F]), axis=(-2,-1)) - np.nanmean(BT * A * np.expand_dims(ZF, (-2,-1)), axis=(0,1))[...,0]

Q = ZF / Z2
Q_ = Q - ((B @ H * A) @ V)[...,0]

In [190]:
H @ V

array([-0.95063159,  0.05541488, -0.09148283])

In [191]:
show_map(Q_, vmin=-500, vmax=500, sine_lat=True)

In [197]:
nx = len(Q)
#lat = (np.arange(nx) / (nx - 1) * 2 - 1) * 90
lat = np.arcsin(np.arange(nx) / (nx - 1) * 2 - 1) * 180 / np.pi
#flux = np.nanmean(data, axis=1)
flux = np.nanmean(Q * Z2, axis=1) / np.nanmean(Z2, axis=1)

plt.figure(figsize=(10,8))
plt.plot(lat, flux)

plt.xlim(-90,90)
plt.ylim(-10,10)
plt.grid(True)
plt.tight_layout()

/tmp/ipykernel_98661/2559170801.py:5: RuntimeWarning: Mean of empty slice
  flux = np.nanmean(Q * Z2, axis=1) / np.nanmean(Z2, axis=1)


In [194]:
file = '/home/ulyanov/data/sdo/hmi/synoptic_maps/hmi.Synoptic_Mr_720s.2314.synopMr.fits'

with fits.open(file) as hdul:
    header = hdul[0].header.copy()
    data_ = hdul[0].data.copy()

data_ = rebin(data_, 2)
#data_ = gaussian_filter(data_, 2, truncate=3)
header['T_OBS']

'2026.08.16_03:27:27_TAI'

In [195]:
show_map(data_, vmin=-100, vmax=100, sine_lat=True, cmap='seismic')

In [198]:
flux_ = np.nanmean(data_ * Z2, axis=1) / np.nanmean(Z2, axis=1)
#flux_ = np.nanmean(data_, axis=1)

nx, ny = data_.shape
lat_ = np.arcsin(np.arange(nx) / (nx - 1) * 2 - 1) * 180 / np.pi

plt.figure(figsize=(10,8))
plt.plot(lat_, flux_, label='HMI')
plt.plot(lat, flux, label='FDT')

plt.xlabel('Latitude, degrees')
plt.ylabel(r'Flux, Mx/cm$^2$')
plt.xlim(-90,90)

plt.xlim(-90,90)
plt.ylim(-10,10)
plt.grid(True)
plt.legend()
plt.tight_layout()

/tmp/ipykernel_98661/3755918721.py:1: RuntimeWarning: Mean of empty slice
  flux_ = np.nanmean(data_ * Z2, axis=1) / np.nanmean(Z2, axis=1)
